In [22]:
%run "Super_Category_View.ipynb"

289
9361205
9457362
1746457
final estimates is 2312353204.6079636
289
468
468
230599.0
Alpha_MP
Alpha   657,527,266
HL       35,487,747
MP      186,037,882
Name: AMOUNT, dtype: float64
289
9361205
9457362
1746457
Consumption depth: 3
Estimations depth: 3


In [23]:
# df_consumption.head()

In [24]:
# df_redemption.head()

In [25]:
# df_estimations.head()

In [26]:
# df_search_roins.head()

In [27]:
consumption_bu = df_consumption.groupby(level='BU').sum()

consumption_bu_view = (
    consumption_bu.T
    .groupby(level=['Alpha/MP/HL', 'Ad Group'])
    .sum()
    .T
)

Ads_Type = ['Alpha', 'MP', 'HL']
consumption_bu_view = consumption_bu_view.reindex(columns=Ads_Type, level=0)

original_BU = row_index.get_level_values('BU').unique()
consumption_bu_view = consumption_bu_view.reindex(original_BU).fillna(0)


In [28]:
redemption_bu = df_redemption.groupby(level='BU').sum()

redemption_bu_view = (
    redemption_bu.T
    .groupby(level=['Alpha/MP/HL', 'Ad Group'])
    .sum()
    .T
)

Ads_Type = ['Alpha', 'MP', 'HL']
redemption_bu_view = redemption_bu_view.reindex(columns=Ads_Type, level=0)

original_BU = row_index.get_level_values('BU').unique()
redemption_bu_view = redemption_bu_view.reindex(original_BU).fillna(0)


In [29]:
Estimates_bu = df_estimations.groupby(level='BU').sum()

Estimates_bu_view = (
    Estimates_bu.T
    .groupby(level=['Alpha/MP/HL', 'Ad Group'])
    .sum()
    .T
)

Ads_Type = ['Alpha', 'MP', 'HL']
Estimates_bu_view = Estimates_bu_view.reindex(columns=Ads_Type, level=0)

original_BU = row_index.get_level_values('BU').unique()
Estimates_bu_view = Estimates_bu_view.reindex(original_BU).fillna(0)


In [32]:
search_roins_bu = df_search_roins.groupby(level='BU').sum()

search_roins_bu_view = (
    search_roins_bu.T
    .groupby(level=[0, 1])  # Uses position instead of names
    .sum()
    .T
)

Ads_Type = ['Alpha', 'MP', 'HL']
# We use try/except or verify levels to avoid errors if a level is missing
search_roins_bu_view = search_roins_bu_view.reindex(columns=Ads_Type, level=0)

original_BU = row_index.get_level_values('BU').unique()
search_roins_bu_view = search_roins_bu_view.reindex(original_BU).fillna(0)


In [34]:
# print(search_roins_bu_view)

In [36]:
import pandas as pd

# --- 1. Helper Function to Add Grand Total ---
def add_grand_total(df):
    """Calculates the sum of each column and adds it as a 'Grand Total' row."""
    df_with_total = df.copy()
    # sum(numeric_only=True) ensures we don't try to sum strings
    df_with_total.loc[('Grand Total', ''), :] = df_with_total.sum()
    return df_with_total

# --- 2. Create the BU Level Dataframes ---
def get_bu_view_with_total(df):
    """Collapses SuperCat to BU level and adds Grand Total."""
    # Collapse Rows
    bu_raw = df.groupby(level='BU').sum()
    # Collapse Columns (Display/Search) - Using level indices to avoid errors
    bu_view = bu_raw.T.groupby(level=[0, 1]).sum().T
    # Reorder
    main_headers = ['Alpha', 'MP', 'HL']
    existing = [h for h in main_headers if h in bu_view.columns.levels[0]]
    bu_view = bu_view.reindex(columns=existing, level=0)
    # Add Grand Total Row
    bu_view.loc['Grand Total'] = bu_view.sum()
    return bu_view

# --- 3. Setup Data for Both Sheets ---
# Super Category Tables
super_cat_tables = [
    (add_grand_total(df_consumption), "CONSUMPTION DATA"),
    (add_grand_total(df_redemption), "REDEMPTION DATA"),
    (add_grand_total(df_estimations), "ESTIMATION DATA"),
    (add_grand_total(df_search_roins), "SEARCH ROINS")
]

# BU Level Tables
bu_level_tables = [
    (get_bu_view_with_total(consumption_bu_view), "BU CONSUMPTION"),
    (get_bu_view_with_total(redemption_bu_view), "BU REDEMPTION"),
    (get_bu_view_with_total(Estimates_bu_view), "BU ESTIMATION"),
    (get_bu_view_with_total(search_roins_bu_view), "BU SEARCH ROINS")
]

# --- 4. Setup Excel Writer and Formatting ---
file_path = "Analysis_Consumption.xlsx"
writer = pd.ExcelWriter(file_path, engine='xlsxwriter')
workbook = writer.book

# Formats
title_format = workbook.add_format({'bold': True, 'align': 'center', 'font_size': 14, 'bg_color': '#4472C4', 'font_color': 'white', 'border': 2})
thin_format = workbook.add_format({'border': 1})
thick_border_val = 2

def write_and_format(df_list, sheet_name):
    worksheet = workbook.add_worksheet(sheet_name)
    current_col = 0
    gap = 2 
    
    for df, title in df_list:
        n_header_rows = df.columns.nlevels
        n_index_cols = df.index.nlevels
        total_cols = len(df.columns) + n_index_cols
        total_rows = len(df.index) + n_header_rows + 1
        
        # Write Title
        worksheet.merge_range(0, current_col, 0, current_col + total_cols - 1, title, title_format)
        
        # Write DataFrame
        df.to_excel(writer, sheet_name=sheet_name, startrow=1, startcol=current_col)
        
        # Gridlines/Borders
        worksheet.conditional_format(0, current_col, total_rows - 1, current_col + total_cols - 1, {'type': 'no_errors', 'format': thin_format})
        
        # Thick Outer Borders
        worksheet.conditional_format(0, current_col, 0, current_col + total_cols - 1, {'type': 'no_errors', 'format': workbook.add_format({'top': thick_border_val})})
        worksheet.conditional_format(total_rows - 1, current_col, total_rows - 1, current_col + total_cols - 1, {'type': 'no_errors', 'format': workbook.add_format({'bottom': thick_border_val})})
        worksheet.conditional_format(0, current_col, total_rows - 1, current_col, {'type': 'no_errors', 'format': workbook.add_format({'left': thick_border_val})})
        worksheet.conditional_format(0, current_col + total_cols - 1, total_rows - 1, current_col + total_cols - 1, {'type': 'no_errors', 'format': workbook.add_format({'right': thick_border_val})})

        current_col += total_cols + gap
    
    worksheet.hide_gridlines(2)
    worksheet.set_column(0, current_col, 16)

# --- 5. Run and Save ---
write_and_format(super_cat_tables, 'SuperCategoryView')
write_and_format(bu_level_tables, 'BU_Level_View')

writer.close()
